<font size=5>**Brain region horizontal correlation**

In [ ]:
%reset -f
import pandas as pd
AgeDf = pd.read_csv(r"/path/to/your/data\CamCan\KRR parcel size to age\Weights\All nets weights.csv")
AgeDf

In [ ]:
MeanWeightsMap = AgeDf.mean(axis=1)
MeanWeightsMap

In [ ]:
AllMeanWeights = pd.DataFrame()
AllMeanWeights['Age'] = MeanWeightsMap
AllMeanWeights

In [ ]:
# Load cognitive task data
cogTasks = ['Fluid intelligence','Emotion expression recognition','PicturePriming','Face recognition','FamousFaces','Motor learning']
for ct in cogTasks:
    currTaskDf = pd.read_csv(rf'/path/to/your/data\CamCan\KRR parcel size to cognition\weights\results/{ct}/All nets weights.csv')
    MeanWeightsMap = currTaskDf.mean(axis=1)
    AllMeanWeights[ct] = MeanWeightsMap
AllMeanWeights['Motor learning'] = -AllMeanWeights['Motor learning']
AllMeanWeights

In [8]:
import seaborn as sns
import matplotlib.pyplot as plt
WeightCorr = AllMeanWeights.corr()
WeightCorrMasked = WeightCorr.copy().iloc[1:,:-1]

In [ ]:
import numpy as np
plt.rcParams['font.family'] = ['Times New Roman','SimSun']
# plt.rcParams['font.sans-serif'][0]='Arial'
# print(plt.rcParams['font.sans-serif'])
# Create lower triangular matrix
mask = np.triu(np.ones_like(WeightCorrMasked, dtype=bool),1)
# np.fill_diagonal(mask, True)
fig , ax = plt.subplots(figsize =(3,2.7),dpi=300)
# colormap = sns.diverging_palette(220, 10, as_cmap = True)
colormap = sns.color_palette("vlag", as_cmap = True)
YTickNames = ['Gf','EER','PPR','FR','FF','ML']
XTickNames = ['Age','Gf','EER','PPR','FR','FF']
_ = sns.heatmap(
    WeightCorrMasked,
    xticklabels=XTickNames,
    yticklabels=YTickNames,
    cmap = colormap,
    square=True, 
    cbar_kws={'shrink':.85}, 
    ax=ax,
    annot=True, 
    linewidths=0.1,vmax=0.98, linecolor='white',
    annot_kws={'fontsize':7 },
    mask=mask
)
# ax.set_xticklabels(ax.get_xticklabels(),rotation=0,rotation_mode='anchor',ha = 'right',va='center',fontsize=7)
ax.set_yticklabels(ax.get_yticklabels(),rotation=0)
ax.tick_params(axis='both',labelsize=7,length=3,width=1)
cbar=ax.collections[0].colorbar
cbar.ax.tick_params(axis='y',length=0,labelsize = 7,pad=2)
cbar.set_ticks([WeightCorrMasked.min().min(),0.98])
# plt.title('Correlation of weights on parcel level',size=9)
plt.savefig('Parcel level weights heatmap.svg',format='svg')

<font size=5>**Cluster-level correlation**</font>

In [ ]:
import numpy as np
AgeDf = pd.read_csv(r"/path/to/your/data\CamCan\KRR parcel size to age\Weights\All nets nucbynet mean mat.csv",index_col=0)
AgeWeightsVec = np.array(AgeDf)
AgeWeightsVec = AgeWeightsVec.flatten()
index = [f'{y}_{x}' for x in AgeDf.index for y in AgeDf.columns]
NucLevelWeights = pd.DataFrame(index=index)
NucLevelWeights['Age'] = AgeWeightsVec

In [ ]:
# Replication of each cognitive task
cogTasks = ['Fluid intelligence','Emotion expression recognition','PicturePriming','Face recognition','FamousFaces','Motor learning']
for ct in cogTasks:
    CurrDf = pd.read_csv(f"results/{ct}/All nets nucbynet mean mat.csv",index_col=0)
    CurrWeightsVec = np.array(CurrDf)
    CurrWeightsVec = CurrWeightsVec.flatten()
    NucLevelWeights[ct] = CurrWeightsVec
NucLevelWeights['Motor learning']*=-1

In [ ]:
# Create lower triangular matrix
mask = np.triu(np.ones_like(NucLevelWeights.corr(), dtype=bool))
np.fill_diagonal(mask, True)
_ , ax = plt.subplots(figsize =(7, 6),dpi=600)
colormap = sns.diverging_palette(220, 10, as_cmap = True)

YTickNames = ['','Fluid intelligence','Emotion recognition','Picture priming','Face recognition','Famous faces','Motor learning']
XTickNames = ['Age','Fluid intelligence','Emotion recognition','Picture priming','Face recognition','Famous faces']
_ = sns.heatmap(
    NucLevelWeights.corr(),
    xticklabels=XTickNames,
    yticklabels=YTickNames,
    cmap = colormap,
    square=True, 
    cbar_kws={'shrink':.9 }, 
    ax=ax,
    annot=True, 
    linewidths=0.1,vmax=1.0, linecolor='white',
    annot_kws={'fontsize':12 },
    mask=mask
)
ax.set_xticklabels(ax.get_xticklabels(),rotation=45,rotation_mode='anchor',ha = 'right',va='center')
plt.title('Pearson Correlation of Weights in Nuclei Level', y=1.05, size=15)
plt.savefig('figure/Nuclei level weights corr.pdf',format='pdf')

<font size=5>**Change the scatter plot for Age and FI**

In [14]:
from scipy.stats import pearsonr

In [ ]:
# Brain region level
ParcelAge2MlDf = AllMeanWeights[['Age','Fluid intelligence']]
r,p = pearsonr(ParcelAge2MlDf['Age'], ParcelAge2MlDf['Fluid intelligence'])
plt.figure(figsize=(3,2.7),dpi = 300)
sns.regplot(x=ParcelAge2MlDf['Age'],y=ParcelAge2MlDf['Fluid intelligence'],scatter_kws={"color":"purple","s": 5,"zorder":0,'edgecolor':'none'},line_kws={"color":"blue","lw": 1,"zorder":1})
# plt.title('Parcel level weights', size=9)
if p>=0.0001:
    text = f"r = {r:.3f}\np = {p:.3e}"  # Format: 3 decimal places, scientific notation
else:
    text = f"r = {r:.3f}\np < 0.0001"
ax=plt.gca()
ax.text(
    x=0.75, y=0.95,  # Position parameters (relative coordinates, 0~1)
    s=text,
    transform=ax.transAxes,  # Key parameter: use relative coordinate system
    fontsize=7,
    color='black',
    va='top',  # Vertical alignment
    ha='left',  # Horizontal alignment
    bbox=dict(facecolor='white', alpha=0.8, edgecolor='none')  # Background box
)
ax = plt.gca()
ax.xaxis.label.set_size(7)
ax.yaxis.label.set_size(7)
ax.set_xlabel('Age prediction model weights', size=7)
ax.set_ylabel('Gf prediction model weights', size=7)
ax.spines['bottom'].set_linewidth(1)
ax.spines['left'].set_linewidth(1)
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
ax.tick_params(axis='both',width=1,length = 3,labelsize=7)
plt.savefig('Parcel level weights scatter.svg', format='svg')

In [ ]:
NucAge2MlDf = NucLevelWeights[['Age','Fluid intelligence']]
NucAge2MlDf['Nucleus'] = [NucAge2MlDf.index.str.split("_",expand=True)[i][1] for i in range(len(NucAge2MlDf.index))]

In [ ]:
NucAge2MlDf = NucLevelWeights[['Age','Fluid intelligence']]
NucAge2MlDf['Nucleus'] = [NucAge2MlDf.index.str.split("_",expand=True)[i][1] for i in range(len(NucAge2MlDf.index))]
r,p = pearsonr(NucAge2MlDf['Age'],NucAge2MlDf['Fluid intelligence'])
plt.figure(figsize=(6,5),dpi = 300)
sns.regplot(x=NucAge2MlDf['Age'],y=NucAge2MlDf['Fluid intelligence'],scatter=False,line_kws={"color":"blue","lw": 2, "zorder": 1})
sns.scatterplot(x=NucAge2MlDf['Age'],y=NucAge2MlDf['Fluid intelligence'],hue=NucAge2MlDf['Nucleus'],zorder=2)
plt.title('Nuclei Level Weights', size=15)
ax = plt.gca()
if p>=0.0001:
    text = f"r = {r:.3f}\np = {p:.3e}"  # Format: 3 decimal places, scientific notation
else:
    text = f"r = {r:.3f}\np < 0.0001"
ax.text(
    x=0.75, y=0.95,  # Position parameters (relative coordinates, 0~1)
    s=text,
    transform=ax.transAxes,  # Key parameter: use relative coordinate system
    fontsize=12,
    color='black',
    va='top',  # Vertical alignment
    ha='left',  # Horizontal alignment
    bbox=dict(facecolor='white', alpha=0.8, edgecolor='none')  # Background box
)
ax.xaxis.label.set_size(12)
ax.yaxis.label.set_size(12)
ax.spines['bottom'].set_linewidth(2)
ax.spines['left'].set_linewidth(2)
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
ax.tick_params(axis='both',width=2,length = 5)